# Gold: hospital_daily_summary

Builds `ws_healthcare_cntrl_india.gold.hospital_daily_summary` (one row per hospital per day) by aggregating silver lab_results, enriched with hospitals and calendar, with a LAG-based day-over-day trend. Per STTM `sttm_silver_to_gold.xlsx` (sheet `hospital_daily_summary`); full rebuild.

In [0]:
%sql
CREATE OR REPLACE TABLE ws_healthcare_cntrl_india.gold.hospital_daily_summary
COMMENT 'Gold hospital daily summary: one row per hospital per test day with test volumes, abnormal/critical rates and day-over-day trend. Source: STTM silver_to_gold, sheet hospital_daily_summary.'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                 AS summary_date,
    COUNT(*)                                  AS total_tests,
    COUNT(DISTINCT patient_id)                AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                 AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count
  FROM ws_healthcare_cntrl_india.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  w.hospital_id,
  h.hospital_name,
  w.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(w.total_tests AS INT)                                                AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                     AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                        AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                        AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                        AS critical_test_count,
  CAST(w.abnormal_test_count * 100.0 / w.total_tests AS DECIMAL(5,2))       AS abnormal_rate_pct,
  CAST(w.critical_test_count * 100.0 / w.total_tests AS DECIMAL(5,2))       AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                       AS prev_day_total_tests,
  CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
       ELSE CAST((w.total_tests - w.prev_day_total_tests) * 100.0 / w.prev_day_total_tests AS DECIMAL(6,2))
  END                                                                       AS test_volume_change_pct
FROM with_lag w
LEFT JOIN ws_healthcare_cntrl_india.silver.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN ws_healthcare_cntrl_india.silver.calendar c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT);

In [0]:
# Return the gold table record count to the caller (e.g. a Job or dbutils.notebook.run)
updated_df = spark.table("ws_healthcare_cntrl_india.gold.hospital_daily_summary")
record_count = updated_df.count()
print(f"gold.hospital_daily_summary record count: {record_count}")

dbutils.notebook.exit(str(record_count))